# Convert GIFTI surfaces to PLY and visualize them

This notebook finds every `.gii` surface in this folder, extracts its point and triangle arrays, writes a matching `.ply` mesh, validates the result, rebuilds cleaner meshes from the binary hippocampus volumes, and compares the raw and rebuilt surfaces in an interactive Plotly view.

Required packages: `nibabel`, `numpy`, `pandas`, `trimesh`, `plotly`, `scipy`, and `scikit-image`.


In [7]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import trimesh
from IPython.display import display

# The explicit path makes the notebook work even if Jupyter was started elsewhere.
# Fall back to the current directory if the folder is moved with the notebook.
DATA_DIR = Path("/home/jakaria/INR/Deep3DComp/adni_hipp_surfs")
if not DATA_DIR.is_dir():
    DATA_DIR = Path.cwd()

gii_files = sorted(DATA_DIR.glob("*.gii"))
if not gii_files:
    raise FileNotFoundError(f"No .gii files found in {DATA_DIR}")

print(f"Surface folder: {DATA_DIR}")
print("GIFTI files:")
for path in gii_files:
    print(f"  - {path.name}")

Surface folder: /home/jakaria/INR/Deep3DComp/adni_hipp_surfs
GIFTI files:
  - 1001_bl.L.hipp.surf.gii
  - 1001_bl.R.hipp.surf.gii


## Load and convert the surfaces

GIFTI surface geometry stores vertices with the `POINTSET` intent and triangle indices with the `TRIANGLE` intent. The checks below prevent an invalid or non-surface GIFTI file from being silently exported.

In [8]:
POINTSET = nib.nifti1.intent_codes["NIFTI_INTENT_POINTSET"]
TRIANGLE = nib.nifti1.intent_codes["NIFTI_INTENT_TRIANGLE"]


def load_gifti_surface(path):
    """Return an (N, 3) vertex array and an (M, 3) triangle array."""
    image = nib.load(str(path))
    pointsets = [d.data for d in image.darrays if d.intent == POINTSET]
    triangles = [d.data for d in image.darrays if d.intent == TRIANGLE]

    if len(pointsets) != 1 or len(triangles) != 1:
        raise ValueError(
            f"{path.name}: expected one POINTSET and one TRIANGLE array; "
            f"found {len(pointsets)} and {len(triangles)}"
        )

    vertices = np.asarray(pointsets[0], dtype=np.float32)
    faces = np.asarray(triangles[0], dtype=np.int64)

    if vertices.ndim != 2 or vertices.shape[1] != 3:
        raise ValueError(f"{path.name}: invalid vertex shape {vertices.shape}")
    if faces.ndim != 2 or faces.shape[1] != 3:
        raise ValueError(f"{path.name}: invalid triangle shape {faces.shape}")
    if not np.isfinite(vertices).all():
        raise ValueError(f"{path.name}: vertices contain NaN or infinite values")
    if faces.size and (faces.min() < 0 or faces.max() >= len(vertices)):
        raise ValueError(f"{path.name}: triangle indices are outside the vertex array")

    return vertices, faces


converted = {}
records = []

for gii_path in gii_files:
    vertices, faces = load_gifti_surface(gii_path)
    ply_path = gii_path.with_suffix(".ply")

    # process=False preserves the GIFTI vertex order and triangle indexing.
    mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
    mesh.export(ply_path, file_type="ply")

    # Read the PLY back from disk so conversion failures are caught immediately.
    checked = trimesh.load_mesh(ply_path, file_type="ply", process=False)
    if len(checked.vertices) != len(vertices) or len(checked.faces) != len(faces):
        raise RuntimeError(f"PLY validation failed for {ply_path.name}")

    converted[ply_path.name] = checked
    records.append(
        {
            "input": gii_path.name,
            "output": ply_path.name,
            "vertices": len(vertices),
            "triangles": len(faces),
            "watertight": bool(checked.is_watertight),
            "size_kB": round(ply_path.stat().st_size / 1024, 1),
        }
    )

summary = pd.DataFrame(records)
display(summary)
print(f"\nWrote {len(converted)} PLY file(s) to {DATA_DIR}")

,input,output,vertices,triangles,watertight,size_kB
0,1001_bl.L.hipp.surf.gii,1001_bl.L.hipp.surf.ply,3448,6960,False,129.0
1,1001_bl.R.hipp.surf.gii,1001_bl.R.hipp.surf.ply,2843,5732,False,106.3



Wrote 2 PLY file(s) to /home/jakaria/INR/Deep3DComp/adni_hipp_surfs


## Rebuild the surface from the hippocampus volume and visualize it

The raw `.gii` meshes are open and can produce spikes or holes when smoothed directly. The next cell rebuilds each surface from the corresponding binary `.mgz` volume by keeping the largest voxel component, closing small gaps, filling holes, applying Gaussian smoothing in voxel space, and extracting a new mesh with marching cubes. The final cell compares the raw converted meshes against the volume-remeshed meshes.

In [9]:
from scipy import ndimage
from skimage import measure

GAUSSIAN_SIGMA = 1.2
CLOSING_ITERATIONS = 1


def mgz_path_for_surface_name(surface_name):
    lower = surface_name.lower()
    if '.l.' in lower or lower.startswith('lh'):
        return DATA_DIR / '1001_bl.L.hipp.mgz'
    if '.r.' in lower or lower.startswith('rh'):
        return DATA_DIR / '1001_bl.R.hipp.mgz'
    raise ValueError(f'Cannot infer left/right volume for {surface_name}')


def keep_largest_component(volume_mask):
    labels, component_count = ndimage.label(volume_mask)
    if component_count == 0:
        raise ValueError('The volume mask is empty.')
    if component_count == 1:
        return volume_mask, component_count
    sizes = ndimage.sum(volume_mask, labels, range(1, component_count + 1))
    keep_label = int(np.argmax(sizes)) + 1
    return labels == keep_label, component_count


processed = {}
processing_records = []

for name, raw_mesh in converted.items():
    mgz_path = mgz_path_for_surface_name(name)
    image = nib.load(str(mgz_path))
    volume_mask = np.asarray(image.get_fdata()) > 0

    volume_mask, source_component_count = keep_largest_component(volume_mask)
    volume_mask = ndimage.binary_closing(volume_mask, iterations=CLOSING_ITERATIONS)
    volume_mask = ndimage.binary_fill_holes(volume_mask)
    volume_mask, repaired_component_count = keep_largest_component(volume_mask)

    smooth_volume = ndimage.gaussian_filter(volume_mask.astype(np.float32), sigma=GAUSSIAN_SIGMA)
    spacing = image.header.get_zooms()[:3]
    vertices, faces, _, _ = measure.marching_cubes(smooth_volume, level=0.5, spacing=spacing)

    rebuilt_mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
    rebuilt_components = sorted(
        rebuilt_mesh.split(only_watertight=False),
        key=lambda mesh: len(mesh.faces),
        reverse=True,
    )
    rebuilt_mesh = rebuilt_components[0].copy()
    rebuilt_mesh.remove_unreferenced_vertices()

    out_path = DATA_DIR / name.replace('.ply', '_volume_remesh.ply')
    rebuilt_mesh.export(out_path)
    processed[name] = rebuilt_mesh

    processing_records.append(
        {
            'input_surface': name,
            'source_volume': mgz_path.name,
            'source_components': source_component_count,
            'post_repair_components': repaired_component_count,
            'rebuilt_components': len(rebuilt_components),
            'sigma': GAUSSIAN_SIGMA,
            'watertight': bool(rebuilt_mesh.is_watertight),
            'euler_number': int(rebuilt_mesh.euler_number),
            'vertices': len(rebuilt_mesh.vertices),
            'triangles': len(rebuilt_mesh.faces),
            'output': out_path.name,
        }
    )

processing_summary = pd.DataFrame(processing_records)
display(processing_summary)


,input_surface,source_volume,source_components,post_repair_components,rebuilt_components,sigma,watertight,euler_number,vertices,triangles,output
0,1001_bl.L.hipp.surf.ply,1001_bl.L.hipp.mgz,1,1,1,1.2,True,2,2370,4736,1001_bl.L.hipp.surf_volume_remesh.ply
1,1001_bl.R.hipp.surf.ply,1001_bl.R.hipp.mgz,1,1,1,1.2,True,2,2074,4144,1001_bl.R.hipp.surf_volume_remesh.ply


In [11]:
from plotly.subplots import make_subplots

colors = ['#2E86DE', '#E67E22', '#27AE60', '#8E44AD']
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=('Raw converted meshes', 'Volume-remeshed meshes'),
)

for (name, raw_mesh), color in zip(converted.items(), colors * ((len(converted) + 3) // 4)):
    rebuilt_mesh = processed[name]

    raw_vertices = np.asarray(raw_mesh.vertices)
    raw_faces = np.asarray(raw_mesh.faces)
    fig.add_trace(
        go.Mesh3d(
            x=raw_vertices[:, 0],
            y=raw_vertices[:, 1],
            z=raw_vertices[:, 2],
            i=raw_faces[:, 0],
            j=raw_faces[:, 1],
            k=raw_faces[:, 2],
            name=f'raw: {name}',
            color=color,
            opacity=1.0,
            flatshading=False,
            lighting=dict(ambient=0.35, diffuse=0.75, specular=0.20, roughness=0.70),
            lightposition=dict(x=100, y=180, z=150),
            hovertemplate=f'raw: {name}<br>x=%{{x:.2f}}<br>y=%{{y:.2f}}<br>z=%{{z:.2f}}<extra></extra>',
            showlegend=True,
        ),
        row=1,
        col=1,
    )

    rebuilt_vertices = np.asarray(rebuilt_mesh.vertices)
    rebuilt_faces = np.asarray(rebuilt_mesh.faces)
    fig.add_trace(
        go.Mesh3d(
            x=rebuilt_vertices[:, 0],
            y=rebuilt_vertices[:, 1],
            z=rebuilt_vertices[:, 2],
            i=rebuilt_faces[:, 0],
            j=rebuilt_faces[:, 1],
            k=rebuilt_faces[:, 2],
            name=f'remeshed: {name}',
            color=color,
            opacity=1.0,
            flatshading=False,
            lighting=dict(ambient=0.40, diffuse=0.78, specular=0.18, roughness=0.78),
            lightposition=dict(x=100, y=180, z=150),
            hovertemplate=f'remeshed: {name}<br>x=%{{x:.2f}}<br>y=%{{y:.2f}}<br>z=%{{z:.2f}}<extra></extra>',
            showlegend=True,
        ),
        row=1,
        col=2,
    )

fig.update_layout(
    title='Hippocampal surfaces: raw converted meshes versus volume-remeshed meshes',
    template='plotly_white',
    width=1400,
    height=700,
    margin=dict(l=0, r=0, t=60, b=0),
    legend=dict(x=0.01, y=0.99),
    scene=dict(aspectmode='data', xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    scene2=dict(aspectmode='data', xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
)

fig.show()
